# 2026 COMP90042 Project Group 141 retrieve=sensor transformer
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

In [1]:
# Install the requirements in Google Colab
!pip install -q -U transformers accelerate datasets evaluate

In [2]:
# Authenticate to Hugging Face
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
# setting device on GPU if available, else CPU
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

Using device: cuda



# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)


In [4]:
# load data from Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
from pathlib import Path

# base data path
data_path = Path("/content/drive/MyDrive/Colab Notebooks/COMP90042NLP/COMP90042_Assignment3/data")

# all file paths
data_path_train_claims = data_path / "train-claims.json"
data_path_evidence = data_path / "evidence.json"
data_path_dev_claims = data_path / "dev-claims.json"
data_path_dev_baseline_claims = data_path / "dev-claims-baseline.json"
data_path_test_claims = data_path / "test-claims-unlabelled.json"


In [6]:
import json

with open(data_path_train_claims) as file:
    train_claims = json.load(file)
# train_claims is a dictionary with clailm_id as key and a dictionary as
# value where it stores claim_text, claim_label and evidences.

with open(data_path_evidence) as file:
    evidence = json.load(file)
# evidence is a dictionary with evidence_id as key and the actual text
# as value.

with open(data_path_dev_claims) as file:
    dev_claims = json.load(file)
# dev_claims is a dictionary with clailm_id as key and a dictionary as
# value where it stores claim_text, claim_label and evidences.

with open(data_path_dev_baseline_claims) as file:
    dev_baseline_claims = json.load(file)
# dev_baseline_claims is a dictionary with clailm_id as key and a dictionary as
# value where it stores claim_text, claim_label and evidences.

with open(data_path_test_claims) as file:
    test_claims = json.load(file)
# test_claims is a dictionary with clailm_id as key and a dictionary as
# value where it stores just claim_text.

In [7]:
import re

def clean_text(text):
    text = text.lower()
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

def process_data(data, retrieve_train=False, classification_train=False):
    """Process claims into task-specific rows.

    - retrieve_train=True:
      output: list of {claim_id, claim_text}
    - classification_train=True:
      output: list of {claim_id, claim_text, label, evidences, gold_input}
    Exactly one mode should be True.
    """
    processed_data = []
    for claim_id, value in data.items():
        claim_text = clean_text(value["claim_text"])

        if retrieve_train:
            processed_data.append({
                "claim_id": claim_id,
                "claim_text": claim_text,
            })
        elif classification_train:
            evidence_ids = list(value.get("evidences", []))
            evidence_texts = [clean_text(evidence[eid]) for eid in evidence_ids if eid in evidence]
            gold_input = (
                f"Claim: {claim_text} \n\nEvidence:\n"
                + "\n".join([f"- {e}" for e in evidence_texts])
            )
            processed_data.append({
                "claim_id": claim_id,
                "claim_text": claim_text,
                "evidences": evidence_ids,
                "label": value.get("claim_label"),
                "gold_input": gold_input,
            })

    return processed_data

def add_gold_input_for_records(records):
    """For list records with evidences, append/build `gold_input` in place."""
    for record in records:
        claim_text = clean_text(record["claim_text"])
        evidence_ids = list(record.get("evidences", []))
        evidence_texts = [clean_text(evidence[eid]) for eid in evidence_ids if eid in evidence]

        gold_input = (
            f"Claim: {claim_text} \n\nEvidence:\n"
            + "\n".join([f"- {e}" for e in evidence_texts])
        )

        record["gold_input"] = gold_input

    return records


# Retrieval training/inference rows.
processed_train_retrieve_data = process_data(train_claims, retrieve_train=True)
processed_dev_retrieve_data = process_data(dev_claims, retrieve_train=True)

# Classification training rows built from gold evidences.
processed_train_data = process_data(train_claims, classification_train=True)
processed_dev_data = process_data(dev_claims, classification_train=True)
# processed_train_data, processed_dev_data and processed_dev_baseline_data are lists
# with different claims stored as a dictionary with claim_id, claim_text, label, evidence_ids
# and gold_input. gold_input is a input format for classification.

processed_test_data = process_data(test_claims, retrieve_train=True)
# processed_test_data is a list with different claims stored as a dictionary with
# just claim_id and claim_text.

In [8]:
all_evidence_ids = list(evidence.keys())
all_evidence_texts = [clean_text(evidence[eid]) for eid in all_evidence_ids]
# all_evidence_ids and all_evidence_texts are coresponding lists of all the evidence passages.

In [9]:
print(f"all_evidence_ids.length = {len(all_evidence_ids)}")
print(f"len(processed_train_data) = {len(processed_train_data)}")
print(f"len(processed_dev_data) = {len(processed_dev_data)}")
print(f"len(processed_test_data) = {len(processed_test_data)}")
print(processed_train_data[0])
print(processed_train_data[0].get("gold_input"))

all_evidence_ids.length = 1208827
len(processed_train_data) = 1228
len(processed_dev_data) = 154
len(processed_test_data) = 153
{'claim_id': 'claim-1937', 'claim_text': 'not only is there no scientific evidence that co2 is a pollutant, higher co2 concentrations actually help ecosystems support more plant and animal life.', 'evidences': ['evidence-442946', 'evidence-1194317', 'evidence-12171'], 'label': 'DISPUTED', 'gold_input': 'Claim: not only is there no scientific evidence that co2 is a pollutant, higher co2 concentrations actually help ecosystems support more plant and animal life. \n\nEvidence:\n- at very high concentrations (100 times atmospheric concentration, or greater), carbon dioxide can be toxic to animal life, so raising the concentration to 10,000 ppm (1%) or higher for several hours will eliminate pests such as whiteflies and spider mites in a greenhouse.\n- plants can grow as much as 50 percent faster in concentrations of 1,000 ppm co 2 when compared with ambient condit

# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)


## 2.1 Evidence Retrieval
Given claim_text, output evidence_ids

In [10]:
# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import copy
import random
import numpy as np
import evaluate
from datasets import Dataset
from transformers import TrainingArguments, Trainer
from transformers import EarlyStoppingCallback
from transformers import DataCollatorWithPadding
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline
from tqdm.auto import tqdm

from torch.utils.data import DataLoader
from sentence_transformers import CrossEncoder, InputExample

### 2.1.1 Stage 1: Find top-k evidence

In [11]:
STAGE1_TOP_N_DEFAULT = 200
RERANK_TOP_K_DEFAULT = 5
STAGE1_CLAIM_BATCH_SIZE = 8

# Build TF-IDF index over all evidence passages
# vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)

evidence_tfidf = vectorizer.fit_transform(all_evidence_texts)

evidence_id_to_text = {
    eid: text for eid, text in zip(all_evidence_ids, all_evidence_texts)
}

In [12]:
# Input: claim_text
# Return: top-k evidence_ids
def retrieve_stage1_candidates(claim_text_list, top_n=STAGE1_TOP_N_DEFAULT, return_scores=False):
    """Top-N evidence ids per claim. claim_text_list is a list of claim strings.

    TF-IDF is L2-normalized by default, so sparse dot product equals cosine similarity.
    Returns a list with one entry per input claim (each entry is a list of evidence ids,
    or (ids, scores) if return_scores is True).
    """
    if not claim_text_list:
        return []

    claim_clean_list = [clean_text(t) for t in claim_text_list]
    claim_vecs = vectorizer.transform(claim_clean_list)
    # sims = (claim_vecs @ evidence_tfidf.T)
    # if hasattr(sims, "toarray"):
    #     sims = sims.toarray()
    # sims = np.asarray(sims, dtype=np.float32)
    sims = np.asarray(cosine_similarity(claim_vecs, evidence_tfidf), dtype=np.float32)

    batch, n_docs = sims.shape
    top_n = min(int(top_n), n_docs)
    idx = np.argpartition(-sims, top_n - 1, axis=1)[:, :top_n]
    row_ix = np.arange(batch)[:, None]
    part = sims[row_ix, idx]
    order = np.argsort(-part, axis=1)
    top_indices = idx[row_ix, order]

    results = []
    for b in range(batch):
        row_idx = top_indices[b]
        top_ids = [all_evidence_ids[i] for i in row_idx]
        if return_scores:
            top_scores = [float(sims[b, i]) for i in row_idx]
            results.append((top_ids, top_scores))
        else:
            results.append(top_ids)
    return results


def build_stage1_candidates_for_processed_list(
    processed_retrieve_list,
    top_n=STAGE1_TOP_N_DEFAULT,
    verbose=False,
    show_progress_bar=True,
    claim_batch_size=STAGE1_CLAIM_BATCH_SIZE,
):
    """Build Stage1 top-N candidates using a processed retrieval list.

    Expected item format in the list: {claim_id, claim_text}.
    """
    topk_map = {}
    proc_list = list(processed_retrieve_list)
    n = len(proc_list)
    batch_starts = list(range(0, n, claim_batch_size))
    iterator = batch_starts
    if show_progress_bar:
        iterator = tqdm(batch_starts, desc="Stage1 retrieval", unit="batch")

    for start in iterator:
        chunk = proc_list[start : start + claim_batch_size]
        claim_ids = [item["claim_id"] for item in chunk]
        claim_texts = [item["claim_text"] for item in chunk]
        batch_out = retrieve_stage1_candidates(claim_texts, top_n=top_n, return_scores=False)
        for claim_id, top_ids in zip(claim_ids, batch_out):
            topk_map[claim_id] = top_ids
            if verbose:
                print(f"[Stage1] claim_id={claim_id} candidates={len(top_ids)}")
    return topk_map

In [13]:
# -------------------------
# Stage1 run on train retrieval list
# -------------------------
stage1_output_path = data_path.parent / "train_stage1_topk.json"
# if we have processed data, load it
if stage1_output_path.exists():
    with open(stage1_output_path, "r", encoding="utf-8") as f:
        train_stage1_topk = json.load(f)
    print("Loaded Stage1 candidates from:", stage1_output_path)
else:
    train_stage1_topk = build_stage1_candidates_for_processed_list(
        processed_train_retrieve_data,
        top_n=STAGE1_TOP_N_DEFAULT,
        verbose=False,
    )
    with open(stage1_output_path, "w", encoding="utf-8") as f:
        json.dump(train_stage1_topk, f, ensure_ascii=False, indent=2)
    print("Saved Stage1 candidates to:", stage1_output_path)

print("Stage1 candidates ready for train claims:", len(train_stage1_topk))

Loaded Stage1 candidates from: /content/drive/MyDrive/Colab Notebooks/COMP90042NLP/COMP90042_Assignment3/train_stage1_topk.json
Stage1 candidates ready for train claims: 1228


### 2.1.2 Stage 2: Reranking top-k evidences

In [14]:
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L6-v2"
batch_size=16
num_epochs=1
learning_rate=2e-5
warmup_ratio=0.1
output_path="./stage2_reranker_ckpt"

In [15]:
def build_stage2_training_examples(
    claims_dict,
    stage1_topk_map,
    negatives_per_claim=4,
    seed=42,
):
    """Create pairwise training data for CrossEncoder reranker.

    Positive pairs: (claim, gold evidence) label=1
    Negative pairs: (claim, non-gold evidence from stage1 candidates) label=0
    """
    rng = random.Random(seed)
    examples = []

    for claim_id, item in claims_dict.items():
        claim_text = clean_text(item["claim_text"])
        gold_ids = set(item.get("evidences", []))
        candidates = stage1_topk_map.get(claim_id, [])

        # positives
        for eid in gold_ids:
            ev_text = evidence_id_to_text.get(eid)
            if ev_text is not None:
                examples.append(InputExample(texts=[claim_text, ev_text], label=1.0))

        # hard negatives from top-N candidates that are not in gold
        hard_neg_ids = [eid for eid in candidates if eid not in gold_ids]
        if len(hard_neg_ids) > negatives_per_claim:
            hard_neg_ids = rng.sample(hard_neg_ids, negatives_per_claim)

        for eid in hard_neg_ids:
            ev_text = evidence_id_to_text.get(eid)
            if ev_text is not None:
                examples.append(InputExample(texts=[claim_text, ev_text], label=0.0))

    return examples

stage2_train_examples = build_stage2_training_examples(
    train_claims,
    train_stage1_topk,
    negatives_per_claim=4,
    seed=42,
)

In [16]:
reranker = CrossEncoder(RERANKER_MODEL_NAME, num_labels=1)
train_loader = DataLoader(stage2_train_examples, shuffle=True, batch_size=batch_size)
warmup_steps = max(1, int(len(train_loader) * num_epochs * warmup_ratio))
reranker.fit(
    train_dataloader=train_loader,
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": learning_rate},
    output_path=output_path,
    show_progress_bar=True,
)




config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Step,Training Loss
500,0.456495


In [17]:
def build_retrieval_predictions(
    processed_retrieve_list,
    top_k=RERANK_TOP_K_DEFAULT,
    stage1_top_n=STAGE1_TOP_N_DEFAULT,
    reranker=None,
    rerank_batch_size=32,
    verbose=True,
    show_progress_bar=True,
):
    """Two phases: (1) `build_stage1_candidates_for_processed_list` for all claims;
    (2) Stage2 rerank and write `evidences`.

    Input: list of {claim_id, claim_text}.
    Output: same list with `evidences` filled in place.
    """
    proc_list = list(processed_retrieve_list)
    n = len(proc_list)

    # Phase 1: Stage1 over all claims (reuse the same batched helper as train cache).
    stage1_topk_map = build_stage1_candidates_for_processed_list(
        proc_list,
        top_n=stage1_top_n,
        verbose=False,
        show_progress_bar=show_progress_bar,
    )

    # Phase 2: Stage2 rerank (per claim; candidate count bounded by stage1_top_n).
    phase2_iter = range(n)
    if show_progress_bar:
        phase2_iter = tqdm(range(n), desc="Stage2 rerank", unit="claim")

    for idx in phase2_iter:
        item = proc_list[idx]
        claim_id = item["claim_id"]
        claim_text = item["claim_text"]
        stage1_ids = stage1_topk_map.get(claim_id, [])

        if reranker is None:
            item["evidences"] = stage1_ids[:top_k]
        else:
            claim_clean = clean_text(claim_text)
            cand_ids = [eid for eid in stage1_ids if eid in evidence_id_to_text]
            pairs = [[claim_clean, evidence_id_to_text[eid]] for eid in cand_ids]
            if not pairs:
                item["evidences"] = stage1_ids[:top_k]
            else:
                scores = reranker.predict(
                    pairs, batch_size=rerank_batch_size, show_progress_bar=False
                )
                ranked = sorted(
                    zip(cand_ids, scores), key=lambda x: float(x[1]), reverse=True
                )
                item["evidences"] = [eid for eid, _ in ranked[:top_k]]

        if verbose:
            mode = "two-stage" if reranker is not None else "stage1-only"
            print(f"[{mode}] claim_id={claim_id} claim_text={claim_text}")
            print(item["evidences"])

    return processed_retrieve_list


# -------- Dev: two-stage retrieval (uses `reranker` from Stage2 finetuning above) --------
stage2_reranker = reranker  # alias for dev/test pipeline cells below

dev_retrieval_top5 = build_retrieval_predictions(
    copy.deepcopy(processed_dev_retrieve_data),
    top_k=RERANK_TOP_K_DEFAULT,
    stage1_top_n=STAGE1_TOP_N_DEFAULT,
    reranker=reranker,
    rerank_batch_size=32,
    verbose=False,
)
print("Dev retrieval done:", len(dev_retrieval_top5), "claims")
if dev_retrieval_top5:
    print("Sample retrieval row:", dev_retrieval_top5[0])


Stage1 retrieval:   0%|          | 0/20 [00:00<?, ?batch/s]

Stage2 rerank:   0%|          | 0/154 [00:00<?, ?claim/s]

Dev retrieval done: 154 claims
Sample retrieval row: {'claim_id': 'claim-752', 'claim_text': '[south australia] has the most expensive electricity in the world.', 'evidences': ['evidence-572512', 'evidence-67732', 'evidence-48256', 'evidence-780332', 'evidence-1061888']}


## 2.2 Classification

In [18]:
# load model
MODEL_NAME = "google-bert/bert-base-uncased"
LABEL_LIST = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]
label2id = {name: i for i, name in enumerate(LABEL_LIST)}
id2label = {i: name for name, i in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
)
model.to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [19]:
# Tokenize
def claims_to_dataset(processed_rows):
    return Dataset.from_dict({
        "text": [row["gold_input"] for row in processed_rows],
        "labels": [label2id[row["label"]] for row in processed_rows],
    })

train_hf = claims_to_dataset(processed_train_data)
dev_hf = claims_to_dataset(processed_dev_data)

def tokenize_batch(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

train_tok = train_hf.map(tokenize_batch, batched=True, remove_columns=["text"])
dev_tok = dev_hf.map(tokenize_batch, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/1228 [00:00<?, ? examples/s]

Map:   0%|          | 0/154 [00:00<?, ? examples/s]

In [20]:


accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)


CLASSIFIER_OUTPUT_DIR = "./bert_classifier_ckpt"
#CLASSIFIER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(CLASSIFIER_OUTPUT_DIR),

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=1,
    # save_strategy="no",
    # load_best_model_at_end=False,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=50,
    seed=42,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=dev_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)
print(f"train on {model.device}")
trainer.train()

train on cuda:0


Epoch,Training Loss,Validation Loss,Accuracy
1,1.182606,0.990993,0.603896
2,0.896154,0.885954,0.649351
3,0.765167,0.907656,0.655844
4,0.674904,0.889492,0.668831
5,0.562178,0.892035,0.707792
6,0.423987,0.901775,0.642857
7,0.387984,0.977839,0.701299
8,0.303340,0.990855,0.681818
9,0.246607,1.038039,0.707792
10,0.198889,1.031016,0.694805


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=770, training_loss=0.5542578303968752, metrics={'train_runtime': 1019.1318, 'train_samples_per_second': 12.049, 'train_steps_per_second': 0.756, 'total_flos': 1873643358879552.0, 'train_loss': 0.5542578303968752, 'epoch': 10.0})

In [21]:
def build_pipeline_predictions(retrieval_records, batch_size=16):
    """Attach BERT claim_label using already-retrieved evidences."""
    # formatted_records: list of {claim_id, claim_text, label, evidences, gold_input}
    formatted_records = add_gold_input_for_records(retrieval_records)

    # Keep deterministic ordering by claim_id before batching/prediction.
    formatted_records = sorted(formatted_records, key=lambda x: x["claim_id"])
    texts = [record["gold_input"] for record in formatted_records]

    clf_pipe = pipeline(
        "text-classification",
        model=model,
        tokenizer=tokenizer,
        truncation=True,
        max_length=512,
        device=device,
    )

    preds = clf_pipe(texts, batch_size=batch_size)
    labels = [pred["label"] for pred in preds]

    return {
        record["claim_id"]: {
            "claim_text": record["claim_text"],
            "claim_label": lab,
            "evidences": list(record["evidences"]),
        }
        for record, lab in zip(formatted_records, labels)
    }


def run_retrieval_then_classification(
    processed_retrieve_records,
    top_k=5,
    batch_size=16,
    verbose=False,
    reranker=None,
    stage1_top_n=STAGE1_TOP_N_DEFAULT,
    rerank_batch_size=32,
    show_progress_bar=True,
):
    """Full pipeline used for unlabeled test claims.

    Input: records with {claim_id, claim_text}.
    Output: {claim_id: {claim_label, evidences}} ready for eval format.
    """
    print(f"Retrieval start:")
    retrieval_records = build_retrieval_predictions(
        processed_retrieve_records,
        top_k=top_k,
        stage1_top_n=stage1_top_n,
        reranker=reranker,
        rerank_batch_size=rerank_batch_size,
        verbose=verbose,
        show_progress_bar=show_progress_bar,
    )
    print(f"Retrieval End:")
    print(f"Classification:")
    return build_pipeline_predictions(retrieval_records, batch_size=batch_size)


# Teacher-forcing sanity check: use gold evidence inputs and compute dev accuracy.
_dev_pred_map = build_pipeline_predictions(processed_dev_data, batch_size=16)
print(
    "Dev accuracy (gold evidence inputs):",
    sum(
        _dev_pred_map[row["claim_id"]]["claim_label"] == dev_claims[row["claim_id"]]["claim_label"]
        for row in processed_dev_data
    )
    / len(processed_dev_data),
)

Dev accuracy (gold evidence inputs): 0.7077922077922078


# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [22]:
import numpy as np


def official_eval_like_script(predictions, groundtruth, verbose=False):
    """Same metrics as eval.py. Returns (F, A, harmonic_mean)."""
    f_scores, acc = [], []
    for claim_id, claim in sorted(groundtruth.items()):
        if claim_id not in predictions:
            continue
        pred = predictions[claim_id]
        if "claim_label" not in pred or "evidences" not in pred:
            continue

        instance_correct = 1.0 if pred["claim_label"] == claim["claim_label"] else 0.0

        evidence_correct = 0
        evidence_recall = 0.0
        evidence_precision = 0.0
        evidence_fscore = 0.0
        if isinstance(pred["evidences"], list) and len(pred["evidences"]) > 0:
            pred_set = set(pred["evidences"])
            for gr_ev in claim["evidences"]:
                if gr_ev in pred_set:
                    evidence_correct += 1
            if evidence_correct > 0:
                evidence_recall = float(evidence_correct) / len(claim["evidences"])
                evidence_precision = float(evidence_correct) / len(pred["evidences"])
                evidence_fscore = (2 * evidence_precision * evidence_recall) / (
                    evidence_precision + evidence_recall
                )

        if verbose:
            print("groundtruth =", claim)
            print("predictions =", pred)
            print("instance accuracy =", instance_correct)
            print("evidence recall =", evidence_recall)
            print("evidence precision =", evidence_precision)
            print("evidence fscore =", evidence_fscore, "\n\n")

        acc.append(instance_correct)
        f_scores.append(evidence_fscore)

    mean_f = float(np.mean(f_scores if len(f_scores) > 0 else [0.0]))
    mean_acc = float(np.mean(acc if len(acc) > 0 else [0.0]))
    if mean_f == 0.0 and mean_acc == 0.0:
        hmean = 0.0
    else:
        hmean = (2 * mean_f * mean_acc) / (mean_f + mean_acc)

    return mean_f, mean_acc, hmean


# `build_pipeline_predictions` and `run_retrieval_then_classification`
# are defined in the earlier inference cell.

# Dev: run full pipeline then evaluate with labels.
dev_pipeline_predictions = run_retrieval_then_classification(
    processed_dev_retrieve_data,
    top_k=5,
    batch_size=16,
    verbose=False,
    reranker=stage2_reranker,
    stage1_top_n=STAGE1_TOP_N_DEFAULT,
    rerank_batch_size=32,
)

F, A, H = official_eval_like_script(
    dev_pipeline_predictions, dev_claims, verbose=False
)
print("Evidence Retrieval F-score (F)    =", F)
print("Claim Classification Accuracy (A) =", A)
print("Harmonic Mean of F and A          =", H)


Retrieval start:


Stage1 retrieval:   0%|          | 0/20 [00:00<?, ?batch/s]

Stage2 rerank:   0%|          | 0/154 [00:00<?, ?claim/s]

Retrieval End:
Classification:
Evidence Retrieval F-score (F)    = 0.18337456194599053
Claim Classification Accuracy (A) = 0.38311688311688313
Harmonic Mean of F and A          = 0.2480316030469776


In [23]:
import json

# Test (unlabeled): run retrieval + classification and save as JSON.
test_predictions = run_retrieval_then_classification(
    processed_test_data,
    top_k=5,
    batch_size=16,
    verbose=False,
    reranker=stage2_reranker,
    stage1_top_n=STAGE1_TOP_N_DEFAULT,
    rerank_batch_size=32,
)

output_path = data_path.parent / "test-output.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

print("Saved test predictions to:", output_path)
print("Number of test claims:", len(test_predictions))

Retrieval start:


Stage1 retrieval:   0%|          | 0/20 [00:00<?, ?batch/s]

Stage2 rerank:   0%|          | 0/153 [00:00<?, ?claim/s]

Retrieval End:
Classification:
Saved test predictions to: /content/drive/MyDrive/Colab Notebooks/COMP90042NLP/COMP90042_Assignment3/test_predictions.json
Number of test claims: 153


## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*